In [29]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# 1. Создаем Spark-сессию в локальном режиме
spark = SparkSession.builder \
    .appName("CovidAnalysis") \
    .master("local[*]") \
    .getOrCreate()

print("✅ Spark успешно запущен!")

✅ Spark успешно запущен!


In [30]:
# 2. Читаем CSV файл
df = (
    spark.read
    .option("header", True)               # Указываем что есть заголовок
    .option("inferSchema", True)          # Spark сам определит, где числа, а где текст
    .csv("file:///home/jovyan/work/covid-data.csv") # Путь к файлу внутри контейнера Jupyter (автоматически при поднятии контейнера jovyan)
    .withColumn("date", F.to_date("date", "yyyy-MM-dd")) # Превращаем текст "2021-03-31" в настоящий тип ДАТА
)


print(f"✅ Всего строк в файле: {df.count()}")
df.select("iso_code", "location", "date", "total_cases", "new_cases", "population").show(5)

✅ Всего строк в файле: 82289
+--------+-----------+----------+-----------+---------+-----------+
|iso_code|   location|      date|total_cases|new_cases| population|
+--------+-----------+----------+-----------+---------+-----------+
|     AFG|Afghanistan|2020-02-24|        1.0|      1.0|3.8928341E7|
|     AFG|Afghanistan|2020-02-25|        1.0|      0.0|3.8928341E7|
|     AFG|Afghanistan|2020-02-26|        1.0|      0.0|3.8928341E7|
|     AFG|Afghanistan|2020-02-27|        1.0|      0.0|3.8928341E7|
|     AFG|Afghanistan|2020-02-28|        1.0|      0.0|3.8928341E7|
+--------+-----------+----------+-----------+---------+-----------+
only showing top 5 rows



In [31]:
#очищаем от мусора по типу World и т.д.

countries_df = df.filter(
    F.col("iso_code").isNotNull() & 
    ~F.col("iso_code").startswith("OWID_")
)

print(f"✅ После очистки осталось строк с реальными странами: {countries_df.count()}")

✅ После очистки осталось строк с реальными странами: 77804


In [32]:
#Выберите 15 стран с наибольшим процентом переболевших на 31 марта (в выходящем датасете необходимы колонки: iso_code, страна, процент переболевших)
task1_result = (
    countries_df
    # Шаг 1: Оставляем только дату 31 марта 2021 года
    .filter(F.col("date") == "2021-03-31")
    
    # Шаг 2: Убираем страны, где нет данных (защита от деления на ноль или null)
    .filter(
        F.col("total_cases").isNotNull() & 
        F.col("population").isNotNull() & 
        (F.col("population") > 0)
    )
    
    # Шаг 3: Считаем процент: (случаи / население) * 100. Округляем до 2 знаков.
    .withColumn(
        "percent_of_ill_people", 
        F.round((F.col("total_cases") / F.col("population")) * 100, 2)
    )
    
    # Шаг 4: Сортируем от большего процента к меньшему
    .orderBy(F.col("percent_of_ill_people").desc())
    
    # Шаг 5: Оставляем только первые 15 строк
    .limit(15)
    
    # Шаг 6: Оставляем только 3 нужные колонки и переименовываем их для красоты
    .select(
        F.col("iso_code"),
        F.col("location").alias("страна"),
        F.col("percent_of_ill_people").alias("процент переболевших")
    )
)

print("--- ЗАДАЧА 1: Топ-15 стран по проценту переболевших ---")
task1_result.show(truncate=False)

--- ЗАДАЧА 1: Топ-15 стран по проценту переболевших ---
+--------+-------------+--------------------+
|iso_code|страна       |процент переболевших|
+--------+-------------+--------------------+
|AND     |Andorra      |15.54               |
|MNE     |Montenegro   |14.52               |
|CZE     |Czechia      |14.31               |
|SMR     |San Marino   |13.94               |
|SVN     |Slovenia     |10.37               |
|LUX     |Luxembourg   |9.85                |
|ISR     |Israel       |9.63                |
|USA     |United States|9.2                 |
|SRB     |Serbia       |8.83                |
|BHR     |Bahrain      |8.49                |
|PAN     |Panama       |8.23                |
|PRT     |Portugal     |8.06                |
|EST     |Estonia      |8.02                |
|SWE     |Sweden       |7.97                |
|LTU     |Lithuania    |7.94                |
+--------+-------------+--------------------+



In [33]:
#Top 10 стран с максимальным зафиксированным кол-вом новых случаев за последнюю неделю марта 2021 в отсортированном порядке по убыванию
#(в выходящем датасете необходимы колонки: число, страна, кол-во новых случаев)
# Шаг 1: Создаем "Окно". 
# Логика окна: "Сгруппируй данные по стране (partitionBy), и внутри каждой страны отсортируй по new_cases по убыванию (orderBy)"
window_max = Window.partitionBy("location").orderBy(F.col("new_cases").desc())

task2_result = (
    countries_df
    # Шаг 2: Оставляем только последнюю неделю марта (с 25 по 31 число)
    .filter(F.col("date").between("2021-03-25", "2021-03-31"))
    
    # Шаг 3: Присваиваем ранг внутри каждой страны. 1 - это день с макс. новыми случаями.
    .withColumn("rank", F.row_number().over(window_max))
    
    # Шаг 4: Оставляем только наибольшие ранги = 1 и убираем строки, где new_cases = null
    .filter((F.col("rank") == 1) & F.col("new_cases").isNotNull())
    
    # Шаг 5: Теперь сортируем получившийся список стран между собой по этому максимуму
    .orderBy(F.col("new_cases").desc())
    
    # Шаг 6: Берем топ-10
    .limit(10)
    
    # Шаг 7: Форматируем вывод
    .select(
        F.col("date").alias("число"),
        F.col("location").alias("страна"),
        F.col("new_cases").cast("long").alias("кол-во новых случаев")
    )
)

print("--- ЗАДАЧА 2: Топ-10 стран по макс. новым случаям за неделю ---")
task2_result.show(truncate=False)

--- ЗАДАЧА 2: Топ-10 стран по макс. новым случаям за неделю ---
+----------+-------------+--------------------+
|число     |страна       |кол-во новых случаев|
+----------+-------------+--------------------+
|2021-03-25|Brazil       |100158              |
|2021-03-26|United States|77321               |
|2021-03-31|India        |72330               |
|2021-03-31|France       |59054               |
|2021-03-31|Turkey       |39302               |
|2021-03-26|Poland       |35145               |
|2021-03-31|Germany      |25014               |
|2021-03-26|Italy        |24076               |
|2021-03-25|Peru         |19206               |
|2021-03-26|Ukraine      |18226               |
+----------+-------------+--------------------+



In [34]:
#Посчитайте изменение случаев относительно предыдущего дня в России за последнюю неделю марта 2021. 
#(например: в россии вчера было 9150 , сегодня 8763, итог: -387) (в выходящем датасете необходимы колонки: 
#число, кол-во новых случаев вчера, кол-во новых случаев сегодня, дельта)

# Шаг 1: Создаем окно, которое просто сортирует данные по дате (без группировки по странам, т.к. страна уже одна)
window_russia_date = Window.orderBy("date")

task3_result = (
    countries_df
    # Шаг 2: Берем Россию матушку
    .filter(F.col("location") == "Russia")
    
    # Шаг 3: Оставляем только нужные колонки
    .select("date", "new_cases")
    
    # Шаг 4: Сдвигаем колонку new_cases на 1 строку вниз. Эмуляция вчерашнего дня
    .withColumn(
        "new_cases_yesterday",
        F.lag("new_cases", 1).over(window_russia_date)
    )
    
    # Шаг 5: Считаем дельту: сегодня - вчера
    .withColumn(
        "delta",
        F.col("new_cases") - F.col("new_cases_yesterday")
    )
    
    # Шаг 6: фильтруем по датам (последняя неделя марта)
    .filter(F.col("date").between("2021-03-25", "2021-03-31"))
    
    # Шаг 7: Сортируем по дате
    .orderBy(F.col("date").asc())
    
    # Шаг 8: Переименовываем колонки и превращаем числа в целые
    .select(
        F.col("date").alias("число"),
        F.col("new_cases_yesterday").cast("long").alias("кол-во новых случаев вчера"),
        F.col("new_cases").cast("long").alias("кол-во новых случаев сегодня"),
        F.col("delta").cast("long").alias("дельта")
    )
)

print("--- ЗАДАЧА 3: Динамика новых случаев в России ---")
task3_result.show(truncate=False)

--- ЗАДАЧА 3: Динамика новых случаев в России ---
+----------+--------------------------+----------------------------+------+
|число     |кол-во новых случаев вчера|кол-во новых случаев сегодня|дельта|
+----------+--------------------------+----------------------------+------+
|2021-03-25|8769                      |9128                        |359   |
|2021-03-26|9128                      |9073                        |-55   |
|2021-03-27|9073                      |8783                        |-290  |
|2021-03-28|8783                      |8979                        |196   |
|2021-03-29|8979                      |8589                        |-390  |
|2021-03-30|8589                      |8162                        |-427  |
|2021-03-31|8162                      |8156                        |-6    |
+----------+--------------------------+----------------------------+------+



In [35]:
# Сохраняем результаты каждой задачи в отдельные CSV файлы

task1_result.coalesce(1).write.csv("/home/jovyan/work/task1_top_15_countries.csv", header=True, mode="overwrite")
task2_result.coalesce(1).write.csv("/home/jovyan/work/task2_top_10_max_cases.csv", header=True, mode="overwrite")
task3_result.coalesce(1).write.csv("/home/jovyan/work/task3_russia_delta.csv", header=True, mode="overwrite")

print("✅ Результаты успешно сохранены в CSV файлы в папке data!")

✅ Результаты успешно сохранены в CSV файлы в папке data!


In [36]:
spark.stop()
print("✅ Сессия Spark завершена.")

✅ Сессия Spark завершена.
